<a href="https://colab.research.google.com/github/anantshri1/low-resource-mt-malayalam/blob/main/Translation_Sidequest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Imports and Initialiation**

In [ ]:
import os
os.environ['PYTHONHASHSEED'] = '0'

In [2]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report,confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import NMF

import random

random.seed(0)
np.random.seed(0)

In [ ]:
import re                                  # library for regular expression operations
import string                              # for string operations

In [ ]:
import tensorflow as tf
tf.random.set_seed(0)

In [ ]:
import nltk

from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

import spacy
from spacy.matcher import Matcher
from spacy.pipeline import EntityRuler

nlp = spacy.load("en_core_web_sm")

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset
from huggingface_hub import list_datasets
from datasets import load_dataset, get_dataset_config_names, concatenate_datasets

In [7]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

**Using AI4Bharat/BPCC**

The ai4bharat/BPCC dataset on HuggingFace is a comprehensive parallel corpus. It includes the IN22 benchmark — a multi-domain, n-way parallel test set across 22 Indic languages with subsets covering news, entertainment, culture, legal, and India-centric topics. This gives you a general-domain English–Kannada foundation to fine-tune from.

In [ ]:
from datasets import load_dataset

bpcc = load_dataset(
    "ai4bharat/BPCC",
    "bpcc-seed-latest",
    trust_remote_code=True,
    streaming=False          # download fully
)

print(bpcc)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ai4bharat/BPCC' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ai4bharat/BPCC' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this

README.md: 0.00B [00:00, ?B/s]

bpcc-seed-latest/asm_Beng.tsv:   0%|          | 0.00/35.8M [00:00<?, ?B/s]

bpcc-seed-latest/ben_Beng.tsv:   0%|          | 0.00/38.2M [00:00<?, ?B/s]

bpcc-seed-latest/brx_Deva.tsv:   0%|          | 0.00/38.0M [00:00<?, ?B/s]

...

Generating sat_Olck split: 0 examples [00:00, ? examples/s]

Generating tam_Taml split: 0 examples [00:00, ? examples/s]

Generating tel_Telu split: 0 examples [00:00, ? examples/s]

Generating urd_Arab split: 0 examples [00:00, ? examples/s]

DatasetDict({
    asm_Beng: Dataset({
        features: ['tgt', 'src', 'src_lang', 'tgt_lang'],
        num_rows: 99490
    })
    ben_Beng: Dataset({
        features: ['tgt', 'src', 'src_lang', 'tgt_lang'],
        num_rows: 107870
    })
    brx_Deva: Dataset({
        features: ['tgt', 'src', 'src_lang', 'tgt_lang'],
        num_rows: 103774
    })
    doi_Deva: Dataset({
        features: ['tgt', 'src', 'src_lang', 'tgt_lang'],
        num_rows: 91128
    })
    gom_Deva: Dataset({
        features: ['tgt', 'src', 'src_lang', 'tgt_lang'],
        num_rows: 98104
    })
    guj_Gujr: Dataset({
        features: ['tgt', 'src', 'src_lang', 'tgt_lang'],
        num_rows: 105694
    })
    hin_Deva: Dataset({
        features: ['tgt', 'src', 'src_lang', 'tgt_lang'],
        num_rows: 96226
    })
    kan_Knda: Dataset({
        features: ['tgt', 'src', 'src_lang', 'tgt_lang'],
        num_rows: 97075
    })
    kas_Arab: Dataset({
        features: ['tgt', 'src', 'src_lang', 'tgt_lang'

In [ ]:
full_data = bpcc["mal_Mlym"]

# Inspect size first
print(f"Total examples: {len(full_data)}")
print(full_data[0])   # confirm fields look right

Total examples: 98031
{'tgt': 'രണ്ട് ദിവസത്തെ ആക്രമണത്തിനുശേഷം ഹൈദർ അലി കാവേരിപട്ടണം പിടിച്ചെടുക്കാൻ നീങ്ങിയപ്പോൾ ചംഗാമയിലെ ബ്രിട്ടീഷ് കമാൻഡറായ  കേണൽ ജോസഫ് സ്മിത്ത് ഒടുക്കം  വിഭവശേഖരണത്തിനും  സൈന്യപോഷണത്തിനുമായി തിരുവണ്ണാമലയിലേയ്ക്ക് പിൻവാങ്ങി.\n', 'src': 'Hyder Ali moved on to capture Kaveripattinam after two days of siege, while the British commander at Changama, Colonel Joseph Smith, eventually retreated to Tiruvannamalai for supplies and reinforcements.', 'src_lang': 'eng_Latn', 'tgt_lang': 'mal_Mlym'}


In [ ]:
split = full_data.train_test_split(test_size=0.2, seed=42)
train_data = split["train"]
val_data   = split["test"]

print(f"Train: {len(train_data)}, Val: {len(val_data)}")


Train: 78424, Val: 19607


**Preprocessing the text**

`pipeline()` internally loads the tokeniser and model, runs inference, and decodes the output. The `model= argument` is a model ID from the HuggingFace Hub.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
import torch

model_id  = "Helsinki-NLP/opus-mt-en-dra"
tokeniser = MarianTokenizer.from_pretrained(model_id)
model     = MarianMTModel.from_pretrained(model_id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

print(f"Device: {device}")


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/818k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.17M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Device: cuda


*Checking if this actually works or not*

In [ ]:
def translate(sentences):
    inputs = tokeniser(
        sentences,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, num_beams=4, max_length=256)

    return tokeniser.batch_decode(output_ids, skip_special_tokens=True)

test = [">>mal<< Where is the nearest hospital?"]
for src, tgt in zip(test, translate(test)):
    print(f"EN: {src}")
    print(f"DR: {tgt}")
    print()

EN: >>mal<< Where is the nearest hospital?
DR: അടുത്തുള്ള ആശുപത്രി എവിടെ?



**Preprocessing (for dummies)**

The model doesn't understand text. It only understands numbers. So preprocessing is just: turn every sentence into a list of integers.
The tokeniser does this. It has a fixed vocabulary — a dictionary mapping words (or parts of words) to integers. For example:

```
"dog"  → 4721
"the"  → 2
"sat"  → 891
```

So the sentence `"the dog sat"` becomes `[2, 4721, 891]`.

```
MAX_LEN = 256  # if a sentence is longer than 256 tokens, chop it off
```

```
def preprocess_batch(batch):
    # batch is just a chunk of your dataset — a dict with keys "src" and "tgt"
    # batch["src"] is a list of English sentences
    # batch["tgt"] is a list of Malayalam sentences

    # >>mal<< is a special word in this tokeniser's vocabulary
    # prepending it to every source sentence tells the model
    # "your job is to output Malayalam"
    src = [f">>mal<< {s}" for s in batch["src"]]
    tgt = batch["tgt"]

    # Tokenise the English (source) sentences
    # This gives you input_ids and attention_mask for each sentence
    model_inputs = tokeniser(
        src,
        max_length=MAX_LEN,
        truncation=True,  # chop anything longer than MAX_LEN
        padding=False,    # don't pad yet — that happens later per batch
    )

    # Tokenise the target sentences
    # text_target= tells the tokeniser these are the expected outputs, not inputs
    labels = tokeniser(
        text_target=tgt,
        max_length=MAX_LEN,
        truncation=True,
        padding=False,
    )

    # Attach the target token IDs as "labels"
    # labels = the correct answer the model is trying to learn to produce
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
```

This renders everything in the dataset as:

```
{
  "input_ids":      [4, 156, 2031, ...],   # English sentence as integers
  "attention_mask": [1, 1, 1, ...],        # 1 = real token, 0 = padding
  "labels":         [892, 44, 7120, ...]   # Kannada sentence as integers
}
```

The ``.map()`` call just applies this function to every example in the dataset, 256 at a time:

```
tokenised[name] = data.map(
    preprocess_batch,
    batched=True,       # process 256 sentences at once, not one by one
    batch_size=256,
    remove_columns=["src", "tgt", "src_lang", "tgt_lang"],  # drop the raw text, keep only numbers
)
```

In [ ]:
MAX_LEN = 256

def preprocess_batch(batch):
    src = [f">>mal<< {s}" for s in batch["src"]]
    tgt = batch["tgt"]

    model_inputs = tokeniser(
        src,
        max_length=MAX_LEN,
        truncation=True,
        padding=False,
    )

    labels = tokeniser(
        text_target=tgt,
        max_length=MAX_LEN,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenised = {}
for name, data in [("train", train_data), ("validation", val_data)]:
    tokenised[name] = data.map(
        preprocess_batch,
        batched=True,
        batch_size=256,
        remove_columns=["src", "tgt", "src_lang", "tgt_lang"],
    )
    print(f"{name}: {len(tokenised[name])} examples")

# Sanity check
print(tokenised["train"][0])

Map:   0%|          | 0/78424 [00:00<?, ? examples/s]

train: 78424 examples


Map:   0%|          | 0/19607 [00:00<?, ? examples/s]

validation: 19607 examples
{'input_ids': [13, 131, 23, 104, 4, 127, 5, 4, 48927, 35165, 4807, 32027, 30288, 49609, 8839, 1894, 4550, 4267, 2, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [1421, 2045, 2065, 8077, 34462, 1450, 300, 785, 24651, 2991, 8016, 10591, 238, 3198, 2572, 5098, 5705, 706, 35714, 47954, 26433, 4928, 2, 0]}


**Fine-tuning the Transformer**

* The data collator

```
pythondata_collator = DataCollatorForSeq2Seq(
    tokeniser,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
)
```
The sentences are all different lengths. But to train efficiently, you process them in batches — and batches need to be rectangular (every row the same length). The collator pads shorter sentences with a special `<PAD>` token to make them all the same length as the longest one in that batch.
`pad_to_multiple_of=8` just rounds up to the nearest multiple of 8 for GPU efficiency.

* The metric function
```
compute_metrics(eval_preds):
    preds, labels = eval_preds
```
`preds` are what the model output. `labels` are what it should have output. Both are batches of integer lists at this point.

 ```
 labels = np.where(labels != -100, labels, tokeniser.pad_token_id)
 ```
During training, padding positions in the labels get replaced with `-100` — a special value that tells the loss function "ignore this position, don't penalise the model for getting padding wrong". Before decoding labels back to text, we need to swap `-100` back to the actual padding token ID.
```
decoded_preds  = tokeniser.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokeniser.batch_decode(labels, skip_special_tokens=True)
```

Convert integer lists back to human-readable strings. `skip_special_tokens=True` removes things like `<PAD>` and `<EOS>` from the output.
```
result = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])
    return {"bleu": round(result.score, 2)}
```
BLEU is the standard metric for translation quality. It measures how much the model's output overlaps with the reference translation. 0 = completely wrong, 100 = perfect match.

* Training arguments
```
training_args = Seq2SeqTrainingArguments(
    output_dir="./en-kn-marian",   # where to save checkpoints
    num_train_epochs=3,            # go through the full dataset 3 times
    per_device_train_batch_size=32, # process 32 sentences at a time
    per_device_eval_batch_size=32,
    warmup_steps=500,              # start with a tiny learning rate, ramp up over 500 steps
                                   # prevents unstable updates at the start
    weight_decay=0.01,             # mild regularisation — discourages the model from
                                   # changing too drastically from its pretrained weights
    learning_rate=5e-5,            # how big each update step is — small because we're
                                   # fine-tuning, not training from scratch
    fp16=True,                     # use 16-bit numbers instead of 32-bit
                                   # halves memory usage, runs faster on GPU
    predict_with_generate=True,    # during evaluation, actually generate translations
                                   # rather than just looking at raw logits
    generation_max_length=256,
    eval_strategy="epoch",         # run evaluation once per epoch
    save_strategy="epoch",         # save a checkpoint once per epoch
    load_best_model_at_end=True,   # when training finishes, restore whichever
                                   # checkpoint had the best BLEU score
    metric_for_best_model="bleu",
    logging_steps=100,             # print a loss update every 100 steps
    report_to="none",              # don't send logs anywhere external
```

* The Trainer
```
trainer = Seq2SeqTrainer(
    model=model,                        # the model to train
    args=training_args,                 # all the settings above
    train_dataset=tokenised["train"],   # your training data
    eval_dataset=tokenised["validation"], # your validation data
    data_collator=data_collator,        # the padding function from earlier
    compute_metrics=compute_metrics,    # the BLEU function from earlier
)

trainer.train()  # actually starts training
```

`Seq2SeqTrainer` is just the manual training loop you saw earlier — the for `batch in dataloader`, `loss.backward()`, `optimiser.step()` cycle — packaged into one object. Calling `.train()` kicks it off.
What you'll see printed every 100 steps is the loss — a number measuring how wrong the model currently is. It should go down over time. At the end of each epoch you'll see the BLEU score on the validation set, which should go up.


In [ ]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.9 MB/s eta 0:00:00


In [ ]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

import sacrebleu

data_collator = DataCollatorForSeq2Seq(
    tokeniser,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = np.where(labels != -100, labels, tokeniser.pad_token_id)
    decoded_preds  = tokeniser.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokeniser.batch_decode(labels, skip_special_tokens=True)
    result = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])
    return {"bleu": round(result.score, 2)}

training_args = Seq2SeqTrainingArguments(
    output_dir="./marian",
    num_train_epochs=3,
    per_device_train_batch_size=32,   # Marian is small, can handle larger batches
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=5e-5,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=256,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    logging_steps=100,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenised["train"],
    eval_dataset=tokenised["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
print(os.listdir("./marian"))

['checkpoint-2451', 'checkpoint-4902', 'checkpoint-7353']


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./marian_continued",   # new output dir — critical
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=0,                    # no warmup — already trained
    weight_decay=0.01,
    learning_rate=2e-5,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=256,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    logging_steps=100,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,   # model is already loaded from checkpoint, don't reload
    args=training_args,
    train_dataset=tokenised["train"],
    eval_dataset=tokenised["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Bleu
1,0.872330,1.328729,19.680000
2,0.924329,1.312344,20.250000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


TrainOutput(global_step=4902, training_loss=0.8363551873423429, metrics={'train_runtime': 2518.8394, 'train_samples_per_second': 62.27, 'train_steps_per_second': 1.946, 'total_flos': 2147258027999232.0, 'train_loss': 0.8363551873423429, 'epoch': 2.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
shutil.copytree("./marian", "/content/drive/MyDrive/marian")

'/content/drive/MyDrive/marian'

In [ ]:
model.save_pretrained("/content/drive/MyDrive/marian_bpcc_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from transformers import MarianTokenizer
tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-dra")
tokenizer.save_pretrained("/content/drive/MyDrive/marian_bpcc_final")


('/content/drive/MyDrive/marian_bpcc_final/tokenizer_config.json',
 '/content/drive/MyDrive/marian_bpcc_final/vocab.json',
 '/content/drive/MyDrive/marian_bpcc_final/source.spm',
 '/content/drive/MyDrive/marian_bpcc_final/target.spm',
 '/content/drive/MyDrive/marian_bpcc_final/added_tokens.json')

In [ ]:
shutil.copytree("./marian_continued", "/content/drive/MyDrive/marian_continued")

FileExistsError: [Errno 17] File exists: '/content/drive/MyDrive/marian_continued'

In [ ]:
print(os.listdir("/content/drive/MyDrive/marian_bpcc_final"))

['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'vocab.json', 'source.spm', 'target.spm']


In [ ]:
from transformers import MarianMTModel, MarianTokenizer

model = MarianMTModel.from_pretrained("/content/drive/MyDrive/marian_bpcc_final")
tokenizer = MarianTokenizer.from_pretrained("/content/drive/MyDrive/marian_bpcc_final")

model = model.to("cuda")

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
print("Tokenizer vocab size:", tokenizer.vocab_size)

# Check what model expects
print("Model encoder embed size:", model.config.vocab_size)
print("Model decoder embed size:", model.config.decoder_vocab_size)

# Check source and target languages
print("Source lang:", tokenizer.supported_language_codes)

Tokenizer vocab size: 62952
Model encoder embed size: 62952
Model decoder embed size: 62952
Source lang: ['>>tel<<', '>>kan<<', '>>mal<<', '>>tam<<']


In [ ]:
def translate(text, model, tokenizer):
    inputs = tokenizer([text], return_tensors="pt", padding=True, truncation=True, max_length=256)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # move inputs to same device as model
    translated = model.generate(**inputs)
    return tokenizer.decode(translated[0], skip_special_tokens=True)

# Test with a simple sentence first
print(translate("The cat sat on the mat.", model, tokenizer))

# Then something technical
print(translate("We propose a novel attention mechanism for neural machine translation.", model, tokenizer))


പൂച്ച മുലയിൽ ഇരുന്നു.
ന്യൂറൽ മെഷീൻ വിവർത്തനത്തിനായി ഞങ്ങൾ ഒരു നോവൽ ശ്രദ്ധാ സംവിധാനം നിർദ്ദേശിക്കുന്നു.


In [ ]:
from datasets import load_dataset
from sacrebleu.metrics import BLEU

bleu = BLEU()

def translate_batch(texts, model, tokenizer, target_lang=">>mal<<", batch_size=32):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = [f"{target_lang} {t}" for t in texts[i:i+batch_size]]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                          truncation=True, max_length=256)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.no_grad():
            translated = model.generate(**inputs)
        decoded = tokenizer.batch_decode(translated, skip_special_tokens=True)
        results.extend(decoded)
    return results


sources = val_data['src']
references = val_data['tgt']

hypotheses = translate_batch(sources, model, tokenizer)

score = bleu.corpus_score(hypotheses, [references])
print(score)


BLEU = 15.48 47.6/20.8/10.4/5.6 (BP = 0.999 ratio = 0.999 hyp_len = 243341 ref_len = 243647)


KeyboardInterrupt: 

In [ ]:
arxiv = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
print(arxiv.features)
print(arxiv[0])

README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

ML-Arxiv-Papers.csv:   0%|          | 0.00/147M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/117592 [00:00<?, ? examples/s]

{'Unnamed: 0.1': Value('int64'), 'Unnamed: 0': Value('float64'), 'title': Value('string'), 'abstract': Value('string')}
{'Unnamed: 0.1': 0, 'Unnamed: 0': 0.0, 'title': 'Learning from compressed observations', 'abstract': '  The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive 

In [ ]:
sample = arxiv.shuffle(seed=42).select(range(500))

sources_en = sample['abstract']
synthetic_ml = translate_batch(sources_en, model, tokenizer)

# Save to Drive immediately
import json
synthetic = [{"src": en, "tgt": ml}
             for en, ml in zip(sources_en, synthetic_ml)]

with open("/content/drive/MyDrive/arxiv_synthetic_en_ml.json", "w") as f:
    json.dump(synthetic, f, ensure_ascii=False)

print("Saved", len(synthetic), "pairs")

Saved 500 pairs


In [ ]:


with open("/content/drive/MyDrive/arxiv_synthetic_en_ml.json") as f:
    data = json.load(f)

print(f"Total pairs: {len(data)}")
print("\nFirst pair:")
print("EN:", data[0]['src'][:200])
print("ML:", data[0]['tgt'][:200])

print("\nTechnical pair example:")
technical = [x for x in data if any(t in x['src'].lower()
             for t in ["attention", "gradient", "embedding", "transformer"])]
print("EN:", technical[0]['src'][:200])
print("ML:", technical[0]['tgt'][:200])

Total pairs: 500

First pair:
EN:   In this paper we propose $\epsilon$-Consistent Mixup ($\epsilon$mu).
$\epsilon$mu is a data-based structural regularization technique that combines
Mixup's linear interpolation with consistency regu
ML: ഈ പ്രബന്ധത്തിൽ, യൂഗാസ്ലിയോൺ ഡോളർ ഡോള്‍ (1,കൺസെൻ്റർ മിക്സപ്പ് ഡോ.) ഞങ്ങൾ നിർദ്ദേശിക്കുന്ന പ്രകാരം, യുവ്റോൺസിസ്റ്റൻ്റെ ഡോളർസിസ്റ്റൻ്റ് മിക്സിംഗ് സിംപ്ലിക്കേഷൻ സിംഗിൻ്റെ ക്രമാതീതമാതൃകതയുമായി സംയോജിതമാക്ക

Technical pair example:
EN:   We introduce two-scale loss functions for use in various gradient descent
algorithms applied to classification problems via deep neural networks. This
new method is generic in the sense that it can 
ML: ഫ്‌ളൈറ്റഡ് ന്യൂറൽ ശൃംഖലകൾ വഴിയുള്ള പ്രശ്നങ്ങൾ വർഗ്ഗീകരണത്തിലേക്ക് കൊണ്ടുവരുന്നതിനുള്ള രണ്ട് നിസാരമായ തോതിലുള്ള നഷ്ട പ്രവർത്തനങ്ങൾ നമ്മൾ പരിചയപ്പെടുത്തുന്നു. അർത്ഥത്തിൽ ഈ പുതിയ രീതി സാധാരണയായി, ഗൾഡർ ന്
